# 威尔金森功率分配器

在本笔记本中，我们将创建一个[威尔金森功率分配器](https://www.microwaves101.com/encyclopedias/wilkinson-power-splitters)，它将一个输入信号分成两个相位相同的输出信号。关于该电路的理论结果在参考文献[1]中有所介绍。在这里，我们将重现下面所示的理想电路，并讨论参考文献[2]中的内容。在本示例中，电路被设计为在 1 GHz 频率下工作。<img src="wilkinson_power_divider.png">- [1] P. Hallbjörner, Microw. Opt. Technol. Lett. 38, 99 (2003)。- [2] Microwaves 101：“威尔金森功率分配器”。

In [ ]:
# standard imports
import matplotlib.pyplot as plt
import numpy as np

import skrf as rf
from skrf.circuit import Circuit

rf.stylely()

In [ ]:
# frequency band
freq = rf.Frequency(start=0, stop=2, npoints=501, unit='GHz')

# characteristic impedance of the ports
z0_ports = 50

# resistor
R = 100
line_resistor = rf.media.DefinedGammaZ0(frequency=freq, z0=R)
resistor = line_resistor.resistor(R, name='resistor')

# branches
z0_branches = np.sqrt(2)*z0_ports
beta = freq.w/rf.constants.c
line_branches = rf.media.DefinedGammaZ0(frequency=freq, z0=z0_branches, gamma=0+beta*1j)

d = line_branches.theta_2_d(90, deg=True)  # @ 90°(lambda/4)@ 1 GHz is ~ 75 mm
branch1 = line_branches.line(d, unit='m', name='branch1')
branch2 = line_branches.line(d, unit='m', name='branch2')

# ports
port1 = Circuit.Port(freq, name='port1', z0=50)
port2 = Circuit.Port(freq, name='port2', z0=50)
port3 = Circuit.Port(freq, name='port3', z0=50)

# Connection setup
#┬Note that the order of appearance of the port in the setup is important
connections = [
           [(port1, 0), (branch1, 0), (branch2, 0)],
           [(port2, 0), (branch1, 1), (resistor, 0)],
           [(port3, 0), (branch2, 1), (resistor, 1)]
        ]

# Building the circuit
C = Circuit(connections)

可以通过可视化电路图来检查电路设置（这需要安装 Python 包 `networkx`）。

In [ ]:
C.plot_graph(network_labels=True, edge_labels=True, port_labels=True, port_fontize=2)

让我们来看一下电路的散射参数：

In [ ]:
fig, (ax1,ax2) = plt.subplots(2, 1, sharex=True)
C.network.plot_s_db(ax=ax1, m=0, n=0,  lw=2)  # S11
C.network.plot_s_db(ax=ax1, m=1, n=1,  lw=2)  # S22
ax1.set_ylim(-90, 0)
C.network.plot_s_db(ax=ax2, m=1, n=0,  lw=2)  # S21
C.network.plot_s_db(ax=ax2, m=2, n=0,  ls='--', lw=2)  # S31
ax2.set_ylim(-4, 0)
fig.suptitle('Ideal Wilkinson Divider @ 1 GHz')

## 电流和电压在之前的示例中，我们没有很多内部端口，因为我们将 N 个端口直接连接到外部端口。但是，在更复杂的场景中，电路可能有很多内部端口。此外，之前的示例都使用了串联连接的元件。随着电路复杂性的增加，不可避免地会出现一些并联连接的元件，这为计算电流和电压带来了新的挑战。幸运的是，scikit-rf 能够很好地处理这些复杂的电路场景。可以计算电路内部端口处的电流和电压。

In [ ]:
power = [1,0,0]
phase = [0,0,0]
C.voltages(power, phase)  # or C2.currents(power, phase)

或者，您可以使用“分路网络”，例如在本例中使用的“T”型网络，来创建只包含成对连接的电路，并执行电流和电压计算。虽然使用分路网络更符合构建电路的“经典”方法，但它在代码可读性、复杂性和计算性能方面存在一些缺点。

In [ ]:
tee1 = line_branches.tee(name='tee1')
tee2 = line_branches.tee(name='tee2')
tee3 = line_branches.tee(name='tee3')

cnx = [
    [(port1, 0), (tee1, 0)],
    [(tee1, 1), (branch1, 0)],
    [(tee1, 2), (branch2, 0)],
    [(branch1, 1), (tee2, 0)],
    [(branch2, 1), (tee3, 0)],
    [(tee2, 2), (resistor, 0)],
    [(tee3, 2), (resistor, 1)],
    [(tee3, 1), (port3, 0)],
    [(tee2, 1), (port2, 0)],
]
C2 = Circuit(cnx)

生成的图形稍微有些拥挤：

In [ ]:
C2.plot_graph(network_labels=True, edge_labels=True, port_labels=True, port_fontize=2)

但结果是一样的：

In [ ]:
C.network == C2.network

In [ ]:
fig, (ax1,ax2) = plt.subplots(2, 1, sharex=True)
C2.network.plot_s_db(ax=ax1, m=0, n=0,  lw=2)  # S11
C2.network.plot_s_db(ax=ax1, m=1, n=1,  lw=2)  # S22
ax1.set_ylim(-90, 0)
C2.network.plot_s_db(ax=ax2, m=1, n=0,  lw=2)  # S21
C2.network.plot_s_db(ax=ax2, m=2, n=0,  ls='--', lw=2)  # S31
ax2.set_ylim(-4, 0)
fig.suptitle('Ideal Wilkinson Divider (2nd way) @ 1 GHz')

并且内部电压和电流也保持一致。

In [ ]:
C2.voltages(power, phase)  # or C2.currents(power, phase)

您可以在专门的示例中找到关于电压和电流计算的更多详细信息。